# Figure 5 --- Mechanisms


In [ ]:
import sys
sys.path.append('../src')

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from Config.config import PATHS
from Utils.utils import GetMeasurements
from Utils.interaction import Performer
from Classes.cognitive_model_agents import (
    PayoffM2, AttendanceM2, FocalRegionAgent
)

In [ ]:
free_parameters = {
    'inverse_temperature': 8,
    'learning_rate': 0.2,
    'bias': 1,
    'forget': 0.8,
    'len_history': 2,
    'delta': 0.4,
    'max_regions': 5,
}

simulation_parameters = {
    'num_rounds': 100,
    'num_episodes': 50,
    'verbose': False
}
df_list = []

list_fixed_parameters = [
    (2, 0.5),
    (3, 0.4), (3, 0.7),
    (4, 0.3), (4, 0.5), (4, 0.8),
]

for num_agents, threshold in list_fixed_parameters:
    fixed_parameters = {
        'num_agents': num_agents,
        'threshold': threshold,
    }
    for model_class in [PayoffM2, AttendanceM2, FocalRegionAgent]:
        df2 = Performer.sim(
            agent_class=model_class,
            fixed_parameters=fixed_parameters,
            free_parameters=free_parameters,
            simulation_parameters=simulation_parameters,
        )
        df2['model'] = model_class.name()
        df_list.append(df2)

df2 = pd.concat(df_list, ignore_index=True)
df2['bounded_score'] = df2['score'] / df2['threshold']

In [ ]:
gm = GetMeasurements(
    data=df2, measures=['bounded_efficiency', 'conditional_entropy', 'gini_index']
)
data = gm.get_measurements()

In [ ]:
from Utils.utils import PPT

path_to_data = PATHS['human_data'] / 'multi-player.csv'
data_human = pd.read_csv(path_to_data)
data_human['model'] = data_human.apply(
    lambda row: f"{row['num_players']}-{row['threshold']:.2f}-{row['group']}", axis=1
)

path_to_data_sim = PATHS['simulated_data'] / 'FS+Payoff+Attendance.csv'
data_sim = pd.read_csv(path_to_data_sim)
data_sim['treatment'] = 'Best fitting model'
data_sim['model'] = 'FS+Payoff+Attendance'

df = PPT.concat_dfs(data_human, data_sim)

plot_df = df.copy()
plot_df['source'] = np.where(
    plot_df['model'] == 'FS+Payoff+Attendance',
    'Best fitting model',
    'Humans',
)
plot_df['score_norm'] = plot_df['score'] / plot_df['threshold']
plot_df.tail()


In [ ]:
# 1. Global Academic Styling
sns.set_theme(style="ticks", font_scale=1.1)
plt.rcParams["font.family"] = "serif"

# 2. Display names / palettes
model_display = {
    'Payoff-M2': 'Adaptive RL on payoffs',
    'Attendance-M2': 'Adaptive RL on attendance',
    'Focal Schemata': 'Cognitive schemata matching',
}
model_order = list(model_display.values())
model_palette = {
    'Adaptive RL on payoffs': '#1f77b4',
    'Adaptive RL on attendance': '#e377c2',
    'Cognitive schemata matching': '#2ca02c',
}

REF_COLOR = '#6C757D'
SOURCE_ORDER = ['Humans', 'Best fitting model']
SOURCE_PALETTE = {
    'Humans': '#264653',
    'Best fitting model': '#E76F51',
}

df2_plot = df2.copy()
df2_plot['model'] = df2_plot['model'].map(model_display)
data_plot = data.copy()
data_plot['model'] = data_plot['model'].map(model_display)

# Single figure: row 1 = mechanisms (2 panels), row 2 = human vs best fit (3 panels)
fig = plt.figure(figsize=(12, 8.5))
gs = gridspec.GridSpec(
    2, 1, figure=fig,
    height_ratios=[1, 1],
    hspace=0.45,
    top=0.95, bottom=0.08, left=0.08, right=0.98,
)
gs_top = gs[0].subgridspec(1, 2, wspace=0.28)
gs_bot = gs[1].subgridspec(1, 3, wspace=0.30)

ax_A = fig.add_subplot(gs_top[0, 0])
ax_B = fig.add_subplot(gs_top[0, 1])
ax_C = fig.add_subplot(gs_bot[0, 0])
ax_D = fig.add_subplot(gs_bot[0, 1])
ax_E = fig.add_subplot(gs_bot[0, 2])

# =====================================================================
# ROW 1 — A: Speed of convergence
# =====================================================================
sns.lineplot(
    x='round', y='bounded_score', hue='model',
    hue_order=model_order, palette=model_palette,
    linewidth=2, errorbar=None, data=df2_plot, ax=ax_A,
    legend=False,
)
ax_A.axhline(
    y=1, color='#555555', linestyle='--',
    linewidth=1.5, label='Optimal Score',
)
ax_A.set_xlabel('Round', fontweight='bold', labelpad=8)
ax_A.set_ylabel('Av. bounded score', fontweight='bold', labelpad=8)
ax_A.set_title(r"$\bf{A.}$ Speed of convergence", loc='left', pad=10, fontsize=12)

# =====================================================================
# ROW 1 — B: Efficiency vs. Inequality
# =====================================================================
sns.scatterplot(
    x='gini_index', y='bounded_efficiency', hue='model',
    hue_order=model_order, palette=model_palette,
    data=data_plot, ax=ax_B,
    legend='brief',
)
ax_B.set_xlabel('Gini index', fontweight='bold', labelpad=8)
ax_B.set_ylabel('Bounded efficiency', fontweight='bold', labelpad=8)
ax_B.set_xlim([-0.01, 0.2])
ax_B.set_title(r"$\bf{B.}$ Efficiency vs. Inequality", loc='left', pad=10, fontsize=12)

model_handles, model_labels = ax_B.get_legend_handles_labels()
ax_B.get_legend().remove()
opt_handle, opt_label = ax_A.get_legend_handles_labels()

ax_B.legend(
    handles=model_handles + opt_handle,
    labels=model_labels + opt_label,
    loc='lower right',
    frameon=True,
    facecolor='white',
    edgecolor='#e0e0e0',
    fontsize=8,
)

# =====================================================================
# ROW 2 — C: Score vs round
# =====================================================================
sns.lineplot(
    x='round', y='score_norm', hue='source',
    data=plot_df, ax=ax_C,
    hue_order=SOURCE_ORDER, palette=SOURCE_PALETTE,
    linewidth=2, errorbar=('ci', 95),
)
ax_C.set_xlabel('Round')
ax_C.set_ylabel('Av. score / threshold')
ax_C.set_title(r"$\bf{C.}$ Score vs round", loc='left', pad=10, fontsize=12)
ax_C.legend(frameon=False, fontsize=8, title=None)

# =====================================================================
# ROW 2 — D: Score vs threshold
# =====================================================================
sns.lineplot(
    x='threshold', y='score', hue='source',
    data=plot_df, ax=ax_D,
    hue_order=SOURCE_ORDER, palette=SOURCE_PALETTE,
    linewidth=2, errorbar=('ci', 95),
)
x_lo, x_hi = ax_D.get_xlim()
ax_D.plot(
    [x_lo, x_hi], [x_lo, x_hi],
    color=REF_COLOR, linestyle='--', linewidth=1.2, alpha=0.8, zorder=0,
    label='Perfect score',
)
ax_D.set_xlim(x_lo, x_hi)
ax_D.set_xlabel("Bar's threshold")
ax_D.set_ylabel('Av. score')
ax_D.set_title(r"$\bf{D.}$ Score vs threshold", loc='left', pad=10, fontsize=12)
ax_D.legend(frameon=False, fontsize=8, title=None)

# =====================================================================
# ROW 2 — E: Score vs number of players
# =====================================================================
sns.lineplot(
    x='num_agents', y='score_norm', hue='source',
    data=plot_df, ax=ax_E,
    hue_order=SOURCE_ORDER, palette=SOURCE_PALETTE,
    linewidth=2, errorbar=('ci', 95),
)
ax_E.set_xlabel('Number of players')
ax_E.set_ylabel('Av. score / threshold')
ax_E.set_title(r"$\bf{E.}$ Score vs number of players", loc='left', pad=10, fontsize=12)
ax_E.set_ylim(ax_D.get_ylim())
ax_C.set_ylim(ax_D.get_ylim())
ax_E.legend(frameon=False, fontsize=8, title=None)

for ax in [ax_A, ax_B, ax_C, ax_D, ax_E]:
    ax.grid(True, linestyle=':', alpha=0.6, color='#cccccc')
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

plt.show()


In [ ]:
fig.savefig(PATHS['exploratory_figures'] / 'Fig5.png', dpi=300, bbox_inches='tight')